In [1]:
#!/usr/bin/env python
# coding: utf-8


import os, time
from datetime import date
import pyfastx
import prettytable
import csv
import re
import regex
from fuzzysearch import find_near_matches
from collections import Counter
from Bio import SeqIO

In [2]:
def read_bar_file(file_path, Reverse=False):
    """
    Read barcode file and optionally return reverse complement of each line.

    :param file_path: Path to the file to be read.
    :param Reverse: Boolean, if True, returns the reverse complement of the sequence.
    :return: A list of lines stripped of whitespace, optionally reverse complemented.
    """
    try:
        with open(file_path, 'r') as file:
            if Reverse:
                lines = [line.strip()[::-1].translate(str.maketrans('ATCG','TAGC')) for line in file if line.strip()]
            else:
                lines = [line.strip() for line in file if line.strip()]
        return lines
    except FileNotFoundError:
        print(f"Error: The file {file_path} does not exist.")
        return []
    except Exception as e:
        print(f"An error occurred: {e}")
        return []


def write_to_files(f1_records, f1_fa_records, bc_records,r1_file, r1fa_file, barcode_file):
    try:
        with open(r1_file, 'a') as r1_filter:
            for rec1 in f1_records:
                r1_filter.write(rec1)

        with open(r1fa_file, 'a') as r1_fa_filter:
            for rec1_fa in f1_fa_records:
                r1_fa_filter.write(rec1_fa)

        with open(barcode_file, 'a') as barcode_fq:
            for record in bc_records:
                barcode_fq.write(record)

    except Exception as e:
        print(f"Error writing to file: {e}")


def write_to_csv(counts, csv_file_path):
    with open(csv_file_path,mode='w',newline='',encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(['Barcord_Forward', 'Number'])

            # 按照 'Number' 字段降序排列
            sorted_atacs=sorted(counts.items(),key=lambda item: item[1],reverse=True)

            for barcord, count in sorted_atacs:
                writer.writerow([barcord, count])


In [3]:
def extract_and_cluster(records, *args):
    bf1, bf2, bf3 = args[0] # Forward
    br1, br2, br3 = args[1] # Reverse,暂未启用
    filter_file, filter_fa_file, bar_file = args[2]

    batch_size=10000
    # 初始化计数器
    barcode_res = Counter()
    total_num = 0
    sum_num = sum_per = 0
    total_atp = len(records)
    start_time = time.time()
    filter_buffer = []
    barcode_buffer = []
    filterfa_buffer = []

    #apply to csv
    barcode_counts = Counter()
    # 构造正则表达式以查找连续的条形码
    PF = f"({'|'.join(bf1)})({'|'.join(bf2)})({'|'.join(bf3)})"
    PR = f"({'|'.join(br3)})({'|'.join(br2)})({'|'.join(br1)})"

    for read in records:
        seq = read.seq
        name = read.name
        qual = read.qual
        # 查找barcode forward
        match_f = re.search(PF, seq)
        if match_f:
            start_f = match_f.start()
            end_f = match_f.end()
            # 在前向条形码之后的序列中查找反向条形码
            # PR = match_f.group(0)[::-1].translate(str.maketrans('ATCG','TAGC'))
            match_r = re.search(PR, seq[end_f:])
            if match_r:
                start_r =  end_f + match_r.start()
                end_r = end_f + match_r.end()

                # 确认正向barcode和反向barcode序列是否互补配对
                if match_f.group(0) == match_r.group(0)[::-1].translate(str.maketrans('ATCG','TAGC')):
                    # 截取各个部分
                    barcode = f"doubleMatch_{match_f.group(0)[21:]}_{match_r.group(0)[:-21]}"
                    f_qual = qual[start_f:end_f]
                    r_qual = qual[start_r:end_r]
                    filter_seq = seq[end_f:start_r]  # 两个条形码之间的序列
                    filter_qual = qual[end_f:start_r]

                    # 将记录添加到缓冲区
                    filter_buffer.append(f"@{name}_{barcode}\n{filter_seq}\n+\n{filter_qual}\n")
                    barcode_buffer.append(f"@{name}_{barcode}\n{barcode}\n+\n{f_qual}_{r_qual}\n")
                    filterfa_buffer.append(f">{name}_{barcode}\n{filter_seq}\n")

                    # 更新计数器
                    barcode_res[end_f] += 1
                    total_num += 1
                    barcode_counts[barcode] += 1
                else:
                    barcode = f"doubleNotMatch_{match_f.group(0)[21:]}_{match_r.group(0)[:-21]}"
                    f_qual = qual[start_f:end_f]
                    r_qual = qual[start_r:end_r]
                    filter_seq = seq[end_f:start_r]  # 两个条形码之间的序列
                    filter_qual = qual[end_f:start_r]

                    # 将记录添加到缓冲区
                    filter_buffer.append(f"@{name}_{barcode}\n{filter_seq}\n+\n{filter_qual}\n")
                    barcode_buffer.append(f"@{name}_{barcode}\n{barcode}\n+\n{f_qual}_{r_qual}\n")
                    filterfa_buffer.append(f">{name}_{barcode}\n{filter_seq}\n")

                    # 更新计数器
                    barcode_res[end_f] += 1
                    total_num += 1
                    barcode_counts[barcode] += 1

                continue
            # 只有正向barcode匹配的
            else:
                rc_barcode = match_f.group(0)[::-1].translate(str.maketrans('ATCG','TAGC'))
                barcode = f"singleF_{match_f.group(0)[21:]}_{rc_barcode[:-21]}"
                f_qual = qual[start_f:end_f]
                filter_seq = seq[end_f:]  # 两个条形码之间的序列
                filter_qual = qual[end_f:]

                # 将记录添加到缓冲区
                filter_buffer.append(f"@{name}_{barcode}\n{filter_seq}\n+\n{filter_qual}\n")
                barcode_buffer.append(f"@{name}_{barcode}\n{barcode}\n+\n{f_qual}\n")
                filterfa_buffer.append(f">{name}_{barcode}\n{filter_seq}\n")

                # 更新计数器
                barcode_res[end_f] += 1
                total_num += 1
                barcode_counts[barcode] += 1

                # 当缓冲区达到指定批量大小时，写入文件
                if len(barcode_buffer) >= batch_size:
                    write_to_files(filter_buffer,filterfa_buffer, barcode_buffer, filter_file, filter_fa_file, bar_file)
                    # 清空缓冲区
                    filter_buffer = []
                    barcode_buffer = []
                    filterfa_buffer = []

        # 查看反向互补barcode单端匹配
        match_r2 = re.search(PR, seq)
        if match_r2:
                start_r =  match_r2.start()
                end_r = match_r2.end()

                # 截取各个部分
                f_barcode = match_r2.group(0)[::-1].translate(str.maketrans('ATCG','TAGC'))
                barcode = f"singleB_{f_barcode[21:]}_{match_r2.group(0)[:-21]}"
                r_qual = qual[start_r:end_r]
                filter_seq = seq[1:start_r]  # 两个条形码之间的序列
                filter_qual = qual[1:start_r]

                # 将记录添加到缓冲区
                filter_buffer.append(f"@{name}_{barcode}\n{filter_seq}\n+\n{filter_qual}\n")
                barcode_buffer.append(f"@{name}_{barcode}\n{barcode}\n+\n{r_qual}\n")
                filterfa_buffer.append(f">{name}_{barcode}\n{filter_seq}\n")

                # 更新计数器
                barcode_res[start_r] += 1
                total_num += 1
                barcode_counts[barcode] += 1

                # 当缓冲区达到指定批量大小时，写入文件
                if len(barcode_buffer) >= batch_size:
                    write_to_files(filter_buffer,filterfa_buffer, barcode_buffer, filter_file, filter_fa_file, bar_file)
                    # 清空缓冲区
                    filter_buffer = []
                    barcode_buffer = []
                    filterfa_buffer = []

    # filterfa_buffer=dict((x[1], x) for x in filterfa_buffer).values()
    filterfa_buffer=filterfa_buffer


    # 处理剩余未写入的记录
    if barcode_buffer:
        write_to_files(filter_buffer,filterfa_buffer, barcode_buffer, filter_file, filter_fa_file, bar_file)
    match_rate = total_num / total_atp * 100 if total_atp > 0 else 0
    print(f"Map[{round(match_rate,2)}%] Processing[{total_num}/{total_atp}]\n")


    #打印表格
    result_table = prettytable.PrettyTable()
    result_table.field_names = ['Type','Number', 'Precent', 'Total']

    for i in sorted(barcode_res.keys()):
        temp=barcode_res[i]
        sum_num += temp
        Per =temp / total_num * 100 if total_num > 0 else 0
        sum_per += Per
        result_table.add_row([f"length_{i}:",temp,f'{round(Per, 4)}%',None])
    result_table.add_row(["Summary:",sum_num,f'{round(sum_per, 4)}%',total_num])
    print(result_table)

    return barcode_counts

In [5]:
if __name__ == "__main__":
    print(time.strftime("%H:%M:%S", time.localtime()))
    start_time = time.time()
    start_day = date.today()
    # 定义output的名称
    ob_name = f"{start_day}_pacbio_hifi"
    suf = time.strftime("%H%M%S", time.localtime())#%m%d
    output_dir = f'./20240529-3_output/{ob_name}_{suf}'
    if not os.path.exists(output_dir):
        os.makedirs(f"{output_dir}", exist_ok=True)

    filter_f = f'{output_dir}/filter.fq'
    filter_fa_f = f'{output_dir}/filter.fa'
    bar_f = f'{output_dir}/barcode.fq'
    barcode_Uniq_file = f'{output_dir}/uniq_barcode.csv'
    print(f'\033[35m extract_and_cluste ==> processing {ob_name}...\033[0m')

    store_dir = "./data"
    #fastq_path = f"/content/drive/MyDrive/Colab Notebooks/CQMU_Single_Virus/Geneus_Sample1.HQ.fastq"
    fastq_path = f"/home/lijuan/single_virus_barcodes/6BatchData/20240529-4.fq"
    fq_file = [record for record in pyfastx.Fastq(fastq_path)]

    bf_name = ["/home/lijuan/single_virus_barcodes/de-barcode/Barcode_f1.txt","/home/lijuan/single_virus_barcodes/de-barcode/Barcode_f2.txt","/home/lijuan/single_virus_barcodes/de-barcode/Barcode_f3.txt"]
    BF1, BF2, BF3 = [read_bar_file(file) for file in bf_name]
    BR1, BR2, BR3 = [read_bar_file(file,Reverse=True) for file in bf_name]

    barcode_counts= extract_and_cluster(fq_file,
                                        [BF1, BF2, BF3],
                                        [BR1, BR2, BR3],
                                        [filter_f,filter_fa_f,bar_f],
                                       )

    print('\033[1;32m write_to_csv ==> processing ...\033[0m')
    write_to_csv(barcode_counts, barcode_Uniq_file)

    end_time = time.time()
    cost  = end_time - start_time
    mins = int(cost / 60)
    secs = int(cost % 60)
    print(f"cost[{mins}m {secs}s]\n")

09:14:00
 extract_and_cluste ==> processing 2024-05-30_pacbio_hifi...
Map[26.09%] Processing[7618/29194]

+--------------+--------+----------+-------+
|     Type     | Number | Precent  | Total |
+--------------+--------+----------+-------+
|  length_12:  |   1    | 0.0131%  |  None |
|  length_23:  |   1    | 0.0131%  |  None |
|  length_31:  |   2    | 0.0263%  |  None |
|  length_41:  |   1    | 0.0131%  |  None |
|  length_43:  |   1    | 0.0131%  |  None |
|  length_44:  |   2    | 0.0263%  |  None |
|  length_45:  |   1    | 0.0131%  |  None |
|  length_47:  |   1    | 0.0131%  |  None |
|  length_48:  |   1    | 0.0131%  |  None |
|  length_50:  |   1    | 0.0131%  |  None |
|  length_51:  |   2    | 0.0263%  |  None |
|  length_53:  |   3    | 0.0394%  |  None |
|  length_54:  |   2    | 0.0263%  |  None |
|  length_55:  |   4    | 0.0525%  |  None |
|  length_56:  |   6    | 0.0788%  |  None |
|  length_57:  |   5    | 0.0656%  |  None |
|  length_58:  |   3    | 0.0394%  |  N

In [6]:
if __name__ == "__main__":
    print(time.strftime("%H:%M:%S", time.localtime()))
    path='/home/lijuan/single_virus_barcodes/6BatchData'
    # filenames = os.listdir(path)
    suffix = '_20240529.fq'
    filenames = [os.path.join(path, f) for f in os.listdir(path) if os.path.isfile(os.path.join(path, f)) and f.endswith(suffix)]
    for fastq_path in filenames:
        start_time = time.time()
        start_day = date.today()
        # 定义output的名称
        ob_name = f"{start_day}_pacbio_hifi"
        suf = time.strftime("%H%M%S", time.localtime())#%m%d
        dir_name=os.path.dirname(fastq_path)
        fq_filename=os.path.basename(fastq_path)
        fq_filename_prefix=fq_filename[:12]
        output_dir = f'{dir_name}/de_barcode/'
        if not os.path.exists(output_dir):
            os.makedirs(f"{output_dir}", exist_ok=True)

        filter_f = f'{output_dir}/{fq_filename_prefix}_filter.fq'
        filter_fa_f = f'{output_dir}/{fq_filename_prefix}_filter.fa'
        bar_f = f'{output_dir}/{fq_filename_prefix}_barcode.fq'
        barcode_Uniq_file = f'{output_dir}/{fq_filename_prefix}_uniq_barcode.csv'
        print(f'\033[35m extract_and_cluste ==> processing {ob_name}...\033[0m')

        store_dir = "./data"
        #fastq_path = f"/content/drive/MyDrive/Colab Notebooks/CQMU_Single_Virus/Geneus_Sample1.HQ.fastq"
        # fastq_path = f"/home/lijuan/single_virus_barcodes/6BatchData/20240529-1.fq"
        fq_file = [record for record in pyfastx.Fastq(fastq_path)]

        bf_name = ["/home/lijuan/single_virus_barcodes/de-barcode/Barcode_f1.txt","/home/lijuan/single_virus_barcodes/de-barcode/Barcode_f2.txt","/home/lijuan/single_virus_barcodes/de-barcode/Barcode_f3.txt"]
        BF1, BF2, BF3 = [read_bar_file(file) for file in bf_name]
        BR1, BR2, BR3 = [read_bar_file(file,Reverse=True) for file in bf_name]

        barcode_counts= extract_and_cluster(fq_file,
                                            [BF1, BF2, BF3],
                                            [BR1, BR2, BR3],
                                            [filter_f,filter_fa_f,bar_f],
                                        )

        print('\033[1;32m write_to_csv ==> processing ...\033[0m')
        write_to_csv(barcode_counts, barcode_Uniq_file)

        end_time = time.time()
        cost  = end_time - start_time
        mins = int(cost / 60)
        secs = int(cost % 60)
        print(f"cost[{mins}m {secs}s]\n")

22:39:04
 extract_and_cluste ==> processing 2024-05-30_pacbio_hifi...
Map[25.51%] Processing[26658/104493]

+--------------+--------+----------+-------+
|     Type     | Number | Precent  | Total |
+--------------+--------+----------+-------+
|  length_6:   |   1    | 0.0038%  |  None |
|  length_12:  |   1    | 0.0038%  |  None |
|  length_23:  |   1    | 0.0038%  |  None |
|  length_25:  |   1    | 0.0038%  |  None |
|  length_27:  |   1    | 0.0038%  |  None |
|  length_28:  |   1    | 0.0038%  |  None |
|  length_29:  |   1    | 0.0038%  |  None |
|  length_31:  |   3    | 0.0113%  |  None |
|  length_32:  |   1    | 0.0038%  |  None |
|  length_33:  |   2    | 0.0075%  |  None |
|  length_34:  |   2    | 0.0075%  |  None |
|  length_35:  |   1    | 0.0038%  |  None |
|  length_37:  |   2    | 0.0075%  |  None |
|  length_38:  |   1    | 0.0038%  |  None |
|  length_40:  |   1    | 0.0038%  |  None |
|  length_41:  |   1    | 0.0038%  |  None |
|  length_43:  |   4    |  0.015%  | 